# SME Post-check — post-approval credit review

Four cells: inputs → configuration → run → write the output.
**One run is one dossier.** There is no batch mode.


In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# WeasyPrint (cell 4) opens pango/cairo through dlopen, and ctypes on macOS does
# not look in Homebrew's directory. Must be set BEFORE weasyprint is imported.
for lib_dir in ("/opt/homebrew/lib", "/usr/local/lib"):
    if Path(lib_dir).is_dir():
        os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", lib_dir)
        break

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

TESTCASE_ID    = "case_demo"

TAX_CODE       = "0201123795"
APPROVAL_DATE  = "2026-04-01"     # date the approval takes effect
POSTCHECK_DATE = "2026-09-15"     # date the review is carried out

CASE_DIR = PROJECT_ROOT / "samples" / TESTCASE_ID

print(f"{TESTCASE_ID} · tax code {TAX_CODE} · reviewed on {POSTCHECK_DATE}")


In [ ]:
from src.config import Config, build_llm
from src.tools._executor import sqlite_executor

config = Config(
    financial_statement_llm = build_llm("MODEL_FINANCIAL_STATEMENT"),
    proposal_llm            = build_llm("MODEL_PROPOSAL"),
    sitevisit_llm           = build_llm("MODEL_SITEVISIT"),
    sitevisit_photo_llm     = build_llm("MODEL_SITEVISIT_PHOTO"),
    commentary_llm          = build_llm("MODEL_COMMENTARY"),

    query_executor = sqlite_executor("samples/dummy_db/postcheck_dummy.sqlite"),

    max_extraction_calls = 40,
    max_photo_calls      = 12,
    max_photo_images     = 12,
)


In [ ]:
from src.pipeline import run_postcheck

result = run_postcheck(
    case_dir       = CASE_DIR,
    config         = config,
    tax_code       = TAX_CODE,
    approval_date  = APPROVAL_DATE,
    postcheck_date = POSTCHECK_DATE,
)

for number, step in enumerate(result.steps, 1):
    print(f"  {number}. {step}")

print("Documents:", len(result.documents))
print("Verdicts:", result.counts)
for finding in result.findings:
    print(f"  {finding.status:18s} {finding.rule_id}  {finding.title}")

In [ ]:
# --- Cell 4: write the run out ---
import json
from datetime import datetime

# run_postcheck already rendered the report, commentary included.
run_dir = PROJECT_ROOT / "logs" / f"{TESTCASE_ID}_{datetime.now():%Y%m%d_%H%M%S}"
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "report.md").write_text(result.report_markdown, encoding="utf-8")
(run_dir / "result.json").write_text(
    json.dumps(result.to_dict(), ensure_ascii=False, indent=2, default=str),
    encoding="utf-8")

# Per criterion: which facts it read and what each one held, for tracing a verdict
# by hand. Calls the function rather than a method on `result`, so it still works
# when `result` came from an earlier run - edit the code, re-run this cell only.
from src.report.trace import criteria_trace
(run_dir / "criteria_trace.json").write_text(
    json.dumps(criteria_trace(result.findings, result.facts),
               ensure_ascii=False, indent=2, default=str),
    encoding="utf-8")

try:
    import markdown as markdown_lib
    from weasyprint import HTML
    from src.utils.report.report_style import (
        REPORT_CSS, highlight_failed_rows, tag_wide_tables)
    body = markdown_lib.markdown(result.report_markdown,
                                 extensions=["tables", "fenced_code"])
    html = ('<html><head><meta charset="utf-8"><style>' + REPORT_CSS
            + "</style></head><body>"
            + highlight_failed_rows(tag_wide_tables(body)) + "</body></html>")
    HTML(string=html).write_pdf(run_dir / "report.pdf")
    print("wrote report.pdf")
except Exception as exc:
    print(f"skipped the PDF ({type(exc).__name__}: {exc}); report.md is complete")

print("output written to:", run_dir)
